<a href="https://colab.research.google.com/github/aaaanwz/aaaanwz/blob/main/twitter%E6%84%9F%E6%83%85%E5%88%86%E6%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvcc -V
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2020 NVIDIA Corporation
Built on Mon_Oct_12_20:09:46_PDT_2020
Cuda compilation tools, release 11.1, V11.1.105
Build cuda_11.1.TC455_06.29190527_0
Tue Jun 21 12:38:36 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.46       Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   71C    P8    11W /  70W |      0MiB / 15109MiB |      0%      Default |
|                               |            

テストデータ用意

In [2]:
!pip install neologdn demoji

In [3]:
import pandas as pd
import neologdn
import demoji

FILENAME='data.csv'
train = pd.read_csv(FILENAME).query('labels == labels')
test = pd.read_csv(FILENAME).query('labels != labels').drop('labels',axis=1)

import re

def trim(text):
    text = neologdn.normalize(text)
    text=re.sub(r'[#＃@＃]\w+',"",text) #remove hashtag and mension
    text=re.sub(r'(https?|http)(:\/\/[-_\.!~*\'()a-zA-Z0-9;\/?:\@&=\+\$,%#]+)', "" ,text) #remove url
    text=re.sub(r'[\(\)【].+?[】\{\}]',"",text)
    text=demoji.replace(string=text, repl="")
    text=re.sub("[�▼]","",text)
    return text

train['text'] = train['text'].map(trim)
train['labels'] = train['labels'].astype(int)
test['text'] = test['text'].map(trim)
train.head(100)

,text,labels
0,退職代行を利用するのはあり?なし?利用する際の注意点も解説!\n\n\n\n\n\n\n\n,1
1,よいしょYoutubeう!\nJava Ⅱ学習編:15ラストやっちゃう?総合課題\n\n\n...,3
2,\n記事タイトル:R言語でパッケージのロードに失敗した際の解決手順\n\n 詳しくはこちら\...,1
3,データミックスの評判・口コミは?講師はガチなデータサイエンティスト?\n\n\n\n\n\n...,1
4,Code Village(コードビレッジ)の評判・口コミが凄い?\n\n\n\n\n\n\n...,1
...,...,...
95,「JavaScript」\nで人気があるツイートです!(最新50件中)\n\n[いいね:7]...,2
96,\nまずはProgateで最初期の入門を乗り越えましょう!\nそのあとの理解を深めるためには...,3
97,\n記事タイトル:PythonでXPathを使ってスクレイピングする手順\n\n 詳しくはこ...,1
98,\n記事タイトル:PostgreSQL14を5手順でインストールする方法\n\n 詳しくはこ...,1


分類

In [4]:
!pip install simpletransformers pyarrow>=6.0.0 fugashi ipadic

In [5]:
from simpletransformers.classification import ClassificationArgs, ClassificationModel
import torch
import logging

logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

args = ClassificationArgs(num_train_epochs=1, overwrite_output_dir=True)
model = ClassificationModel("bert", "cl-tohoku/bert-base-japanese-whole-word-masking", num_labels=4, args=args, use_cuda=torch.cuda.is_available())

Some weights of the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialize

In [6]:
# Train the model
model.train_model(train)
# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(train)

INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.


  0%|          | 0/511 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_train_bert_128_4_2
/usr/local/lib/python3.7/dist-packages/transformers/optimization.py:310: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 0 of 1:   0%|          | 0/64 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_model: Training of bert model complete. Saved to outputs/.
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.


  0%|          | 0/511 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_dev_bert_128_4_2


Running Evaluation:   0%|          | 0/64 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_model:{'mcc': 0.6336686704055358, 'eval_loss': 0.5712130414322019}


In [17]:
print(result)

for entry in wrong_predictions:
  print(entry)

{'mcc': 0.6336686704055358, 'eval_loss': 0.5712130414322019}
{'guid': 13, 'text_a': '退職代行を利用するのはあり?なし?利用する際の注意点も解説!\n\n\n\n\n\n\n\n', 'text_b': None, 'label': 2}
{'guid': 24, 'text_a': '先のことを考えて今やらなければいけないこと/できることをやる。\n誰も予測できないワクワクする体験や成長を追求していきます。\n \n', 'text_b': None, 'label': 2}
{'guid': 25, 'text_a': '多くの人はProgateを勧めているけど、\n僕は効率が悪いと思う。\n\n昔は僕もProgateを楽しんでやっていたが、、、\n………\n\n\n\n', 'text_b': None, 'label': 0}
{'guid': 39, 'text_a': 'おはようございます!\n\n・Instagram投稿作成\n・スクール生のオンラインレッスン\n・プログラミング教材作成\n\n今日も今日とて\u200d\n1週間のスタートダッシュを朝活でブーストかけましょう!\n\n\n\n\n', 'text_b': None, 'label': 2}
{'guid': 43, 'text_a': '今日も早起きPrと読書\n\nPrはマルチカメラとLumetriカラーでの色補正。色補正を書籍の通りにやってますが、実際に自分でアレンジして決めるのは、難易度高すぎ\n\n読書は「バビロンの大富豪の教え」読了\n明日からはまた別の本(マンガ)\n\nLove&amp;Peaceよろしく\n\n\n\n\n', 'text_b': None, 'label': 1}
{'guid': 48, 'text_a': '未経験からのプログラミングあるある、その5\n\nプログラミングスクールに通う\n\nスクールの良し悪しではなく、お金を払って満足してしまうことが問題。\n私もプログラミング書籍を購入して満足し、よく積読してる\nまずはProgateなどで基礎知識をつけてから検討することをオススメ!\n\n', 'text_b': None, 'label': 3}
{'gui

In [19]:
predictions, raw_outputs = model.predict(test['text'].values.tolist())
prediction = pd.read_csv(FILENAME).query('labels != labels').drop('labels',axis=1)
prediction['prediction'] = pd.Series(predictions)

INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.


  0%|          | 0/2330 [00:00<?, ?it/s]

  0%|          | 0/292 [00:00<?, ?it/s]

In [20]:
prediction.head()

,text,prediction
398,テックジム(techgym)の評判・口コミは？就職サポートはある？��\n\nhttps:/...,3.0
399,よいしょYoutube〜〜う！��✨\nJava Ⅱ 学習編：12.配列上書いちゃう？��✨...,3.0
400,あくまでも個人的な意見だけど\nprogateを周回してからテキストに入った方が復習にもなる...,3.0
401,テックギークの評判・口コミは？Shopifyエンジニアは今が狙い目？��\n\nhttps:...,3.0
402,ついにジャズピアノもprogate式に… https://t.co/xe9aGdnbHS,3.0


In [21]:
from google.colab import files
filename = 'predictions.csv'
prediction.to_csv(filename, encoding='utf-8-sig')
files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>